# Machine Learning

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import log_loss, r2_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
import xgboost as xgb
from scipy.stats import (
    uniform, 
    randint,
    loguniform
)
import matplotlib.pyplot as plt
import timm
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, random_split, DataLoader, Subset, ConcatDataset
import copy
from pprint import pprint
import json
import os
import warnings
import lightgbm as lgb
import optuna
from statsmodels.stats.outliers_influence import variance_inflation_factor
import torch.nn.functional as F
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts , CosineAnnealingLR

In [ ]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
initial_df = pd.read_parquet("../data/processed/anime_data_2.parquet")
initial_df.info()

In [ ]:
df = initial_df.head(5278)
real_df = initial_df.tail(72)

print(df.info())
print(real_df.info())

## Input Preparation

Right now, the priority is to reduce dimensions. The plan is the following:
* Reduce the dimensions of the image tensors from 512
* Reduce the dimensions of sentimental analysis tensors from 768
* Reduce the pool of producers and studios into embedded vectors

In [ ]:
df = df.reset_index(drop=True)

all_indices = np.arange(len(df))

train_idx, test_idx = train_test_split(
    all_indices,
    test_size=0.10,
    random_state=42
)

train_idx, val_idx = train_test_split(
    train_idx,
    test_size=0.10,
    random_state=42
)

print(df.index[:5])
print(train_idx[:5])

### Some Continuous Variables

In [ ]:
df.info()

In [ ]:
continuous_features = ['prequel_score', 'prequel_members']

for feature in continuous_features:
    cts_scaler = StandardScaler()

    cts_scaler.fit(
        df.loc[train_idx, [feature]]
    )

    df[feature] = cts_scaler.transform(
        df[[feature]]
    )

    real_df[feature] = cts_scaler.fit_transform(
        real_df[[feature]]
    )

print(df['prequel_score'].describe())
print(df['prequel_members'].describe())
real_df['prequel_members'].describe()

In [ ]:
df = df.fillna({'prequel_score': 0, 'prequel_members': 0, 'prequel_type': ""})
real_df = real_df.fillna({'prequel_score': 0, 'prequel_members': 0, 'prequel_type': ""})
real_df.info()

### Studios & Producers

In [ ]:
studio_to_idx = {"<UNK>": 0}
producer_to_idx = {"<UNK>": 0}

for studios in df.iloc[train_idx]["studios"]:
    for studio in studios:
        if studio not in studio_to_idx:
            studio_to_idx[studio] = len(studio_to_idx)

for producers in df.iloc[train_idx]["producers"]:
    for producer in producers:
        if producer not in producer_to_idx:
            producer_to_idx[producer] = len(producer_to_idx)

n_studios = len(studio_to_idx)
n_producers = len(producer_to_idx)

print("Number of studios:", n_studios)
print("Number of producers:", n_producers)

In [ ]:
def get_studio_indices(studios):
    return [
        studio_to_idx.get(studio, 0)
        for studio in studios
    ]

def get_producer_indices(producers):
    return [
        producer_to_idx.get(producer, 0)
        for producer in producers
    ]

df["studio_idx"] = df["studios"].apply(get_studio_indices)
df["producer_idx"] = df["producers"].apply(get_producer_indices)

real_df["studio_idx"] = real_df["studios"].apply(get_studio_indices)
real_df["producer_idx"] = real_df["producers"].apply(get_producer_indices)

print(df["studio_idx"].head())
print(df["producer_idx"].head())

In [ ]:
def create_embedding_bag_inputs(index_lists):
    flat_indices = []
    offsets = []

    current_offset = 0

    for indices in index_lists:
        offsets.append(current_offset)
        flat_indices.extend(indices)
        current_offset += len(indices)

    return (
        torch.tensor(flat_indices, dtype=torch.long),
        torch.tensor(offsets, dtype=torch.long)
    )


studio_indices, studio_offsets = create_embedding_bag_inputs(
    df["studio_idx"]
)

producer_indices, producer_offsets = create_embedding_bag_inputs(
    df["producer_idx"]
)

real_studio_indices, real_studio_offsets = create_embedding_bag_inputs(
    real_df["studio_idx"]
)

real_producer_indices, real_producer_offsets = create_embedding_bag_inputs(
    real_df["producer_idx"]
)

print("Studio indices:", studio_indices.shape)
print("Studio offsets:", studio_offsets.shape)

print("Producer indices:", producer_indices.shape)
print("Producer offsets:", producer_offsets.shape)

In [ ]:
def split_embedding_bag_inputs(index_lists, train_idx, val_idx, test_idx):
    
    def create_for_rows(rows):
        selected_lists = [index_lists[i] for i in rows]
        return create_embedding_bag_inputs(selected_lists)

    train_indices, train_offsets = create_for_rows(train_idx)
    val_indices, val_offsets = create_for_rows(val_idx)
    test_indices, test_offsets = create_for_rows(test_idx)

    return (
        train_indices, train_offsets,
        val_indices, val_offsets,
        test_indices, test_offsets
    )


(
    studio_indices_train,
    studio_offsets_train,
    studio_indices_val,
    studio_offsets_val,
    studio_indices_test,
    studio_offsets_test
) = split_embedding_bag_inputs(
    df["studio_idx"].tolist(),
    train_idx,
    val_idx,
    test_idx
)


(
    producer_indices_train,
    producer_offsets_train,
    producer_indices_val,
    producer_offsets_val,
    producer_indices_test,
    producer_offsets_test
) = split_embedding_bag_inputs(
    df["producer_idx"].tolist(),
    train_idx,
    val_idx,
    test_idx
)

### Synopsis

In [ ]:
initial_semantic_embeddings = np.load('../data/processed/semantic_embeddings.npy')
print(initial_semantic_embeddings.shape)
semantic_embeddings = initial_semantic_embeddings[:5278]
real_semantic_embeddings = initial_semantic_embeddings[-72:]

In [ ]:
text_scaler = StandardScaler()

text_scaler.fit(
    semantic_embeddings[train_idx]
)

semantic_embeddings_train = text_scaler.transform(
    semantic_embeddings[train_idx]
)

semantic_embeddings_val = text_scaler.transform(
    semantic_embeddings[val_idx]
)

semantic_embeddings_test = text_scaler.transform(
    semantic_embeddings[test_idx]
)

real_semantic_embeddings = text_scaler.transform(
    real_semantic_embeddings
)

print(
    "Train text NaNs:",
    np.isnan(semantic_embeddings_train).sum()
)

print(
    "Val text NaNs:",
    np.isnan(semantic_embeddings_val).sum()
)

print(
    "Test text NaNs:",
    np.isnan(semantic_embeddings_test).sum()
)

print(
    "Train text infs:",
    np.isinf(semantic_embeddings_train).sum()
)

In [ ]:
class LearnedProjector(nn.Module):
    """
    Projects a frozen all-mpnet-base-v2 sentence embedding (768-dim)
    down to a smaller learned representation via a linear layer.

    Note: sentence-transformers' mpnet output is L2-normalized by default
    (normalize_embeddings=True), so no extra normalization is applied here
    on the input side.

    Usage:
        projector = LearnedProjector(in_dim=768, out_dim=64)
        z = projector(x)  # x: (batch, 768) -> z: (batch, 64)
    """
    def __init__(self, in_dim: int = 768, out_dim: int = 64,
                 hidden_dim: int | None = None, dropout: float = 0.1):
        super().__init__()

        if hidden_dim is None:
            self.net = nn.Sequential(
                nn.Linear(in_dim, out_dim),
                nn.LayerNorm(out_dim)
            )
        else:
            self.net = nn.Sequential(
                nn.Linear(in_dim, hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, out_dim),
                nn.LayerNorm(out_dim)
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

projector = LearnedProjector(in_dim=768, out_dim=64)

## Images

In [ ]:
image_data = np.load("../data/processed/image_embeddings.npy")

In [ ]:
image_scaler = StandardScaler()

image_scaler.fit(
    image_data[train_idx]
)

image_train = image_scaler.transform(
    image_data[train_idx]
)

image_val = image_scaler.transform(
    image_data[val_idx]
)

image_test = image_scaler.transform(
    image_data[test_idx]
)

image_real = image_scaler.transform(
    image_data[-72:]
)

print("Image train NaNs:", np.isnan(image_train).sum())
print("Image val NaNs:", np.isnan(image_val).sum())
print("Image test NaNs:", np.isnan(image_test).sum())

In [ ]:
print(df.info())
print(real_df.info())

In [ ]:
X_other_pre = df.drop(columns=['thumbnail', 'members', 'title', 'studio_idx', 'producer_idx', 'mal_id', 'cohort', 'producers', 'genres', 'studios', 'demographics', 'themes', 'score_z', 'wc_z', 'favorites_z', 'dropped_z', 'drop_rate', 'drop_rate_z', 'forum_z'])
X_other_pre = X_other_pre.reset_index(drop=True)
X_other_pre = pd.get_dummies(X_other_pre, columns=['rating'], dtype=int, drop_first=True)
X_other_pre = pd.get_dummies(X_other_pre, columns=['prequel_type'], dtype=int, drop_first=True)
X_other_pre['sequel'] = X_other_pre['sequel'].astype(int)
X_other_pre.columns = X_other_pre.columns.str.replace(' ', '_')
print(X_other_pre.info())

real_other_pre = real_df.drop(columns=['thumbnail', 'members', 'title', 'studio_idx', 'producer_idx', 'mal_id', 'cohort', 'producers', 'genres', 'studios', 'demographics', 'themes', 'score_z', 'wc_z', 'favorites_z', 'dropped_z', 'drop_rate', 'drop_rate_z','forum_z'])
real_other_pre = real_other_pre.reset_index(drop=True)
real_other_pre = pd.get_dummies(real_other_pre, columns=['rating'], dtype=int)
real_other_pre = pd.get_dummies(real_other_pre, columns=['prequel_type'], dtype=int)
real_other_pre['sequel'] = real_other_pre['sequel'].astype(int)
real_other_pre.columns = real_other_pre.columns.str.replace(' ', '_')
print(real_other_pre.info())

### Adaptation

In [ ]:
features = ['adaptation_score', 'adaptation_members']

adaptation_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
    ('scaler', StandardScaler())
])

print(X_other_pre[features].isna().sum())

train_data = X_other_pre.iloc[train_idx][features]
val_data   = X_other_pre.iloc[val_idx][features]
test_data  = X_other_pre.iloc[test_idx][features]
real_data = real_other_pre[features]

adaptation_train = adaptation_pipeline.fit_transform(train_data)
adaptation_val   = adaptation_pipeline.transform(val_data)
adaptation_test  = adaptation_pipeline.transform(test_data)
adaptation_real  = adaptation_pipeline.transform(real_data)

print(adaptation_train)

In [ ]:
X_other_pre.loc[train_idx, 'adaptation_score'] = adaptation_train[:, 0]
X_other_pre.loc[val_idx, 'adaptation_score'] = adaptation_val[:, 0]
X_other_pre.loc[test_idx, 'adaptation_score'] = adaptation_test[:, 0]
real_other_pre['adaptation_score'] = adaptation_real[:, 0]

X_other_pre.loc[train_idx, 'adaptation_members'] = adaptation_train[:, 1]
X_other_pre.loc[val_idx, 'adaptation_members'] = adaptation_val[:, 1]
X_other_pre.loc[test_idx, 'adaptation_members'] = adaptation_test[:, 1]
real_other_pre['adaptation_members'] = adaptation_real[:, 1]

print(X_other_pre.info())
print(real_other_pre.info())

### Multicollinearity Test

In [ ]:
def safe_boolean_vif(df):
    zero_variance_cols = df.columns[df.nunique() <= 1].tolist()
    if zero_variance_cols:
        print(f"Dropped zero-variance columns immediately: {zero_variance_cols}")
        df = df.drop(columns=zero_variance_cols)
        
    X = df.copy()
    X['constant'] = 1.0
    
    vif_data = pd.DataFrame()
    vif_data["feature"] = X.columns
    
    vifs = []
    for i in range(len(X.columns)):
        try:
            val = variance_inflation_factor(X.values, i)
            vifs.append(np.inf if np.isinf(val) or np.isnan(val) else val)
        except ZeroDivisionError:
            vifs.append(np.inf)
            
    vif_data["VIF"] = vifs
    
    return vif_data[vif_data["feature"] != 'constant'].reset_index(drop=True)

In [ ]:
X_other_pre['has_prequel_members'].describe()

In [ ]:
vif_results = safe_boolean_vif(X_other_pre)
print("\nFinal VIF Data:")
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(vif_results) 

We can infer the following:
1. We have to remove the "has_prequel" variables. The main worry is that the neural network won't be able to learn from the imputed zeroes. However, since a zero is extremely rare or impossible for prequel score and prequel members, this should not be a big issue.
2. Prequel types will ruin our feature importance reliability. We can turn this into a "prequel_type_tv" bool instead since the only prequel type in the inference data is TV.

In [ ]:
X_other_pre = X_other_pre.drop(columns=['has_prequel_score', 'has_prequel_members', 'has_prequel_type', 'prequel_type_Movie', 'prequel_type_Music',
                                        'prequel_type_ONA', 'prequel_type_OVA', 'prequel_type_PV', 'prequel_type_Special', 'prequel_type_TV', 'prequel_type_TV_Special'])

real_other_pre = real_other_pre.drop(columns=['has_prequel_score', 'has_prequel_members', 'has_prequel_type', 'prequel_type_'])

X_other_pre["prequel_type_tv"] = (df['prequel_type'] == "TV").astype(int)

X_other_pre["prequel_type_tv"].describe()

Now lets run VIF one more time.

In [ ]:
vif_results = safe_boolean_vif(X_other_pre)
print("\nFinal VIF Data:")
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(vif_results) 

In [ ]:
print(X_other_pre.info())
print(real_other_pre.info())

## Baseline Model (LGB)

For the baseline model, we will not be taking studios and producers into account, as it's hard to make an analogous EmbeddingBag for them.

In [ ]:
X_train = X_other_pre.loc[train_idx]
X_val = X_other_pre.loc[val_idx]
X_test = X_other_pre.loc[test_idx]

print(X_train.isna().any().any())
print(X_val.isna().any().any())
print(X_test.isna().any().any())

In [ ]:
y = df['forum_z'] # change depending on metric

y_train = y[train_idx]
y_val = y[val_idx]
y_test = y[test_idx]

In [ ]:
def gaussian_nll_objective(preds, train_data):
    y = train_data.get_label()
    n_samples = len(y)
    preds = preds.reshape(n_samples, 2)
    mu = preds[:, 0]
    s = preds[:, 1]  # s = log(sigma^2)
    var = np.exp(s)
    
    res = mu - y
    grad_mu = res / var
    grad_s = 0.5 * (1.0 - (res**2) / var)
    
    hess_mu = 1.0 / var
    hess_s = 0.5 * (res**2) / var
    hess_s = np.maximum(hess_s, 1e-4) # Stabilize Hessian
    
    grad = np.vstack((grad_mu, grad_s)).T.flatten()
    hess = np.vstack((hess_mu, hess_s)).T.flatten()
    return grad, hess

def gaussian_nll_metric(preds, train_data):
    y = train_data.get_label()
    n_samples = len(y)
    if len(preds.shape) == 1 or preds.ndim == 1:
        preds = preds.reshape(n_samples, 2)
    mu = preds[:, 0]
    s = preds[:, 1]

    variance = np.exp(s)
    sigma = np.sqrt(variance)

    sigma = np.clip(sigma, 1e-6, None)
    variance = sigma ** 2

    nll = 0.5 * np.log(2 * np.pi * variance) + ((y - mu)**2 / (2 * variance))
    
    return 'gaussian_nll', np.mean(nll), False

In [ ]:
def objective(trial):
    # Base dictionary setup
    params = {
        'num_class': 2,
        'verbosity': -1,
        'objective': gaussian_nll_objective,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.08, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 5),
        'num_leaves': trial.suggest_int('num_leaves', 7, 23),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.1, 0.6),
        'subsample': trial.suggest_float('subsample', 0.6, 0.9),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.1, 50.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 50.0, log=True),
    }

    
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    cv_scores = []
    
    for train_idx, val_idx in kf.split(X_train):
        if hasattr(X_train, "iloc"):
            X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
            X_va, y_va = X_train.iloc[val_idx], y_train.iloc[val_idx]
        else:
            X_tr, y_tr = X_train[train_idx], y_train[train_idx]
            X_va, y_va = X_train[val_idx], y_train[val_idx]
        
        dtrain = lgb.Dataset(X_tr, label=y_tr)
        dval = lgb.Dataset(X_va, label=y_va, reference=dtrain)
        
        # Train booster utilizing the modern LightGBM 4.x signature
        bst = lgb.train(
            params,
            train_set=dtrain,
            num_boost_round=400,
            valid_sets=[dval],
            feval=gaussian_nll_metric,  # FIX: feval is still accepted or can use eval_metric
            callbacks=[lgb.early_stopping(stopping_rounds=15, verbose=False)]
        )
        
        preds = bst.predict(X_va, raw_score=True)
        __, score, __ = gaussian_nll_metric(preds, dval)
        cv_scores.append(score)
        
    return np.mean(cv_scores)


In [ ]:
# study = optuna.create_study(direction='minimize')
# study.optimize(objective, n_trials=50)  # Increase n_trials for real production use

# print("\n Optimization Finished!")
# print(f"Best Gaussian NLL Value: {study.best_value:.4f}")
# print("Best Hyperparameters:")
# for key, value in study.best_params.items():
#     print(f"  {key}: {value}")

# print("\nTraining final model on full training set...")
# best_params = study.best_params
# best_params['num_class'] = 2
# best_params['verbosity'] = -1
# best_params['objective'] = gaussian_nll_objective

# full_train_dataset = lgb.Dataset(X_train, label=y_train)
# final_bst = lgb.train(
#     best_params,
#     train_set=full_train_dataset,
#     num_boost_round=500
# )

# preds = final_bst.predict(X_test, raw_score=True)
# dtest = lgb.Dataset(X_test, label=y_test)
# metric_name, final_test_nll, is_higher_better = gaussian_nll_metric(preds, dtest)
# print(f"Final Test {metric_name}: {final_test_nll:.4f}")

## Fusion Network

In [ ]:
class FusionNetwork(nn.Module):

    def __init__(self):
        super().__init__()

        # # Text projector
        # self.text_projector = LearnedProjector(
        #     in_dim=768,
        #     hidden_dim=128,
        #     out_dim=64,
        #     dropout=0.4
        # )

        # Image branch
        self.image_branch = nn.Sequential(
            nn.Linear(418, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.4)
        )

        # Tabular branch
        self.other_branch = nn.Sequential(
            nn.Linear(81, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.4)
        )

        self.studio_embedding = nn.EmbeddingBag(
            num_embeddings=n_studios,
            embedding_dim=8,
            mode='mean'
        )

        self.producer_embedding = nn.EmbeddingBag(
            num_embeddings=n_producers,
            embedding_dim=8,
            mode='mean'
        )

        # Fusion
        self.fusion = nn.Sequential(
            nn.Linear(64 + 32 + 8 + 8, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 2)
        )

    def forward(self, image, other, studio_indices, studio_offsets, producer_indices, producer_offsets):
        # 1. Check raw inputs
        for name, tensor in [('image', image), ('other', other)]:
            if torch.isnan(tensor).any():
                print(f"NaN detected in raw input: {name}")

        image_features = self.image_branch(image)
        other_features = self.other_branch(other)
        
        # 2. Check branches
        if torch.isnan(image_features).any(): print("NaN in image branch (Check BatchNorm/Batch Size)")
        if torch.isnan(other_features).any(): print("NaN in other branch")

        studio_features = self.studio_embedding(studio_indices, studio_offsets)
        producer_features = self.producer_embedding(producer_indices, producer_offsets)
        
        # 3. Check embeddings
        if torch.isnan(studio_features).any(): print("NaN in studio embeddings")

        combined = torch.cat([image_features, other_features, studio_features, producer_features], dim=1)
        
        out = self.fusion(combined)
        if torch.isnan(out).any(): print("NaN generated inside Fusion layers")

        mean = out[:, 0]      # shape [32], not [32, 0:1]
        raw_std = out[:, 1]   # shape [32]

        std = torch.exp(raw_std) + 1e-6
        norm_dist = torch.distributions.Normal(mean, std)
        
        return norm_dist

In [ ]:
real_other_pre = real_other_pre.drop(columns=['rating_G_-_All_Ages'])
real_other_pre["rating_R+_-_Mild_Nudity"] = 0
real_other_pre["prequel_type_tv"] = real_other_pre["prequel_type_TV"]
real_other_pre = real_other_pre.drop(columns=['prequel_type_TV'])

In [ ]:
assert list(X_other_pre.columns) == list(real_other_pre.columns), \
    set(X_other_pre.columns) ^ set(real_other_pre.columns)

In [ ]:
# X_text_train = torch.from_numpy(
#     semantic_embeddings_train.astype(np.float32)
# )

# X_text_val = torch.from_numpy(
#     semantic_embeddings_val.astype(np.float32)
# )

# X_text_test = torch.from_numpy(
#     semantic_embeddings_test.astype(np.float32)
# )

X_image_train = torch.from_numpy(
    image_train.astype(np.float32)
)

X_image_val = torch.from_numpy(
    image_val.astype(np.float32)
)

X_image_test = torch.from_numpy(
    image_test.astype(np.float32)
)

X_other = torch.from_numpy(
    X_other_pre.to_numpy(dtype=np.float32)
)

y_score = torch.tensor(
    df["forum_z"].to_numpy(dtype=np.float32) # change depending on metric
)

# real_text = torch.from_numpy(
#     real_semantic_embeddings.astype(np.float32)
# )

real_image = torch.from_numpy(
    image_real.astype(np.float32)
)

real_other = torch.tensor(
    real_other_pre.to_numpy(dtype=np.float32)
)

In [ ]:
other_train = X_other[train_idx]
other_val = X_other[val_idx]
other_test = X_other[test_idx]

score_train = y_score[train_idx]
score_val = y_score[val_idx]
score_test = y_score[test_idx]

In [ ]:
model = FusionNetwork()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)


class AnimeDataset(torch.utils.data.Dataset):

    def __init__(
        self,
        image,
        other,
        target,
        studio_indices,
        studio_offsets,
        producer_indices,
        producer_offsets
    ):
        self.image = image
        self.other = other
        self.target = target

        self.studio_indices = studio_indices
        self.studio_offsets = studio_offsets

        self.producer_indices = producer_indices
        self.producer_offsets = producer_offsets

    def __len__(self):
        return len(self.target)

    def __getitem__(self, idx):

        # Figure out where this anime's studio list starts
        studio_start = self.studio_offsets[idx]

        if idx + 1 < len(self.studio_offsets):
            studio_end = self.studio_offsets[idx + 1]
        else:
            studio_end = len(self.studio_indices)

        studio_indices = self.studio_indices[
            studio_start:studio_end
        ]

        # Same thing for producers
        producer_start = self.producer_offsets[idx]

        if idx + 1 < len(self.producer_offsets):
            producer_end = self.producer_offsets[idx + 1]
        else:
            producer_end = len(self.producer_indices)

        producer_indices = self.producer_indices[
            producer_start:producer_end
        ]

        return (
            self.image[idx],
            self.other[idx],
            studio_indices,
            producer_indices,
            self.target[idx]
        )

def collate_fn(batch):

    images = torch.stack([item[0] for item in batch])
    others = torch.stack([item[1] for item in batch])
    targets = torch.stack([item[4] for item in batch])

    studio_indices = []
    studio_offsets = []

    current_offset = 0

    for item in batch:
        indices = item[2]

        studio_offsets.append(current_offset)

        studio_indices.extend(
            indices.tolist()
        )

        current_offset += len(indices)

    studio_indices = torch.tensor(
        studio_indices,
        dtype=torch.long
    )

    studio_offsets = torch.tensor(
        studio_offsets,
        dtype=torch.long
    )

    producer_indices = []
    producer_offsets = []

    current_offset = 0

    for item in batch:
        indices = item[3]

        producer_offsets.append(current_offset)

        producer_indices.extend(
            indices.tolist()
        )

        current_offset += len(indices)

    producer_indices = torch.tensor(
        producer_indices,
        dtype=torch.long
    )

    producer_offsets = torch.tensor(
        producer_offsets,
        dtype=torch.long
    )

    return (
        images,
        others,
        studio_indices,
        studio_offsets,
        producer_indices,
        producer_offsets,
        targets
    )

In [ ]:
train_dataset = AnimeDataset(
    X_image_train,
    other_train,
    score_train,
    studio_indices_train,
    studio_offsets_train,
    producer_indices_train,
    producer_offsets_train
)

val_dataset = AnimeDataset(
    X_image_val,
    other_val,
    score_val,
    studio_indices_val,
    studio_offsets_val,
    producer_indices_val,
    producer_offsets_val
)

test_dataset = AnimeDataset(
    X_image_test,
    other_test,
    score_test,
    studio_indices_test,
    studio_offsets_test,
    producer_indices_test,
    producer_offsets_test
)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn
)


In [ ]:
def nll_loss(dist, target):
    # print(dist.log_prob(target).shape)
    return -dist.log_prob(target).mean()

In [ ]:
# full_dataset = ConcatDataset([train_dataset, val_dataset])
# n_samples = len(full_dataset)

# k = 5
# kfold = KFold(n_splits=k, shuffle=True, random_state=42)

# batch_size = train_loader.batch_size
# collate_fn = train_loader.collate_fn 
# patience = 15
# n_epochs = 100

# fold_results = []          
# fold_model_states = []    

# for fold, (train_idx, val_idx) in enumerate(kfold.split(np.arange(n_samples))):

#     print(f"\n===== Fold {fold + 1}/{k} =====")

#     model = FusionNetwork().to(device)  
#     optimizer = torch.optim.AdamW(
#         model.parameters(),
#         lr=2e-4,
#         weight_decay=1e-2
#     )

#     train_subset = Subset(full_dataset, train_idx)
#     val_subset = Subset(full_dataset, val_idx)

#     fold_train_loader = DataLoader(
#         train_subset,
#         batch_size=batch_size,
#         shuffle=True,
#         collate_fn=collate_fn
#     )
#     fold_val_loader = DataLoader(
#         val_subset,
#         batch_size=batch_size,
#         shuffle=False,
#         collate_fn=collate_fn
#     )

#     best_val_loss = float("inf")
#     patience_counter = 0
#     best_model_state = None

#     for epoch in range(n_epochs):

#         model.train()
#         train_loss = 0

#         for (
#             image, other,
#             studio_indices, studio_offsets,
#             producer_indices, producer_offsets,
#             target
#         ) in fold_train_loader:

#             image = image.to(device)
#             other = other.to(device)
#             studio_indices = studio_indices.to(device)
#             studio_offsets = studio_offsets.to(device)
#             producer_indices = producer_indices.to(device)
#             producer_offsets = producer_offsets.to(device)
#             target = target.to(device)

#             prediction = model(
#                 image, other,
#                 studio_indices, studio_offsets,
#                 producer_indices, producer_offsets
#             )

#             loss = nll_loss(prediction, target)

#             optimizer.zero_grad()
#             loss.backward()
#             optimizer.step()

#             train_loss += loss.item()

#         avg_train_loss = train_loss / len(fold_train_loader)

#         model.eval()
#         val_loss = 0

#         with torch.no_grad():
#             for (
#                 image, other,
#                 studio_indices, studio_offsets,
#                 producer_indices, producer_offsets,
#                 target
#             ) in fold_val_loader:

#                 image = image.to(device)
#                 other = other.to(device)
#                 studio_indices = studio_indices.to(device)
#                 studio_offsets = studio_offsets.to(device)
#                 producer_indices = producer_indices.to(device)
#                 producer_offsets = producer_offsets.to(device)
#                 target = target.to(device)

#                 prediction = model(
#                     image, other,
#                     studio_indices, studio_offsets,
#                     producer_indices, producer_offsets
#                 )

#                 loss = nll_loss(prediction, target)
#                 val_loss += loss.item()

#         avg_val_loss = val_loss / len(fold_val_loader)

#         if avg_val_loss < best_val_loss:
#             best_val_loss = avg_val_loss
#             patience_counter = 0
#             best_model_state = copy.deepcopy(model.state_dict())
#         else:
#             patience_counter += 1
#             if patience_counter >= patience:
#                 print(f"Fold {fold + 1} early stopping at epoch {epoch}.")
#                 break

#         if epoch % 5 == 0 or patience_counter == 0:
#             print(
#                 f"  Epoch {epoch}: "
#                 f"Train Loss: {avg_train_loss:.4f} | "
#                 f"Val Loss: {avg_val_loss:.4f}"
#             )

#     print(f"Fold {fold + 1} best val loss: {best_val_loss:.4f}")
#     fold_results.append(best_val_loss)
#     fold_model_states.append(best_model_state)

# fold_results = np.array(fold_results)
# print(f"\n===== CV Results ({k}-fold) =====")
# print(f"Per-fold val loss: {fold_results}")
# print(f"Mean: {fold_results.mean():.4f}  |  Std: {fold_results.std():.4f}")

# best_fold_idx = fold_results.argmin()
# best_model_state = fold_model_states[best_fold_idx]
# model = FusionNetwork().to(device)
# model.load_state_dict(best_model_state)
# print(f"\nLoaded weights from fold {best_fold_idx + 1} (val loss {fold_results[best_fold_idx]:.4f})")

In [ ]:
# model.eval()

# total_nll = 0
# total_samples = 0

# all_predictions = []
# all_targets = []

# with torch.no_grad():

#     for (
#         image,
#         other,
#         studio_indices,
#         studio_offsets,
#         producer_indices,
#         producer_offsets,
#         target
#     ) in test_loader:

#         image = image.to(device)
#         other = other.to(device)

#         studio_indices = studio_indices.to(device)
#         studio_offsets = studio_offsets.to(device)

#         producer_indices = producer_indices.to(device)
#         producer_offsets = producer_offsets.to(device)

#         target = target.to(device)

#         predictions = model(
#             image,
#             other,
#             studio_indices,
#             studio_offsets,
#             producer_indices,
#             producer_offsets
#         )

#         nll = nll_loss(predictions, target)

#         total_nll += nll * 32
#         total_samples += target.size(0)



# final_nll = (
#     total_nll /
#     total_samples
# )


# print(f"Test NLL:  {final_nll:.4f}")

In [ ]:
# def get_full_test_batch(dataset):
#     """Pull the entire test_dataset through the existing collate_fn in one shot."""
#     items = [dataset[i] for i in range(len(dataset))]
#     return collate_fn(items)

# (
#     image_t, other_t,
#     studio_idx_t, studio_off_t,
#     producer_idx_t, producer_off_t,
#     target_t
# ) = get_full_test_batch(test_dataset)

# image_t = image_t.to(device)
# other_t = other_t.to(device)
# studio_idx_t = studio_idx_t.to(device)
# studio_off_t = studio_off_t.to(device)
# producer_idx_t = producer_idx_t.to(device)
# producer_off_t = producer_off_t.to(device)
# target_t = target_t.to(device)

# model.eval()

# @torch.no_grad()
# def eval_nll(image, other, studio_idx, studio_off, producer_idx, producer_off, target):
#     dist = model(image, other, studio_idx, studio_off, producer_idx, producer_off)
#     return nll_loss(dist, target).item()

# baseline_nll = eval_nll(
#     image_t, other_t,
#     studio_idx_t, studio_off_t,
#     producer_idx_t, producer_off_t,
#     target_t
# )
# print(f"Baseline test NLL: {baseline_nll:.4f}")

In [ ]:
def permute_rows(tensor, generator):
    """Row-shuffle a (N, ...) tensor along dim 0, breaking its link to the target."""
    perm = torch.randperm(tensor.size(0), generator=generator, device=tensor.device)
    return tensor[perm]

def permutation_importance_nll(eval_fn, baseline, n_repeats=10, seed=42):
    """
    eval_fn: callable(generator) -> permuted NLL for one repeat
    Returns (mean_delta, std_delta) of NLL increase over n_repeats shuffles.
    """
    g = torch.Generator(device=device)
    deltas = []

    for r in range(n_repeats):
        g.manual_seed(seed + r)
        permuted_nll = eval_fn(g)
        deltas.append(permuted_nll - baseline)

    deltas = np.array(deltas)
    return deltas.mean(), deltas.std()

In [ ]:
# # --- Block-level importance: image branch, tabular branch, studios, producers ---

# block_results = {}

# def eval_permuted_image(g):
#     return eval_nll(
#         permute_rows(image_t, g), other_t,
#         studio_idx_t, studio_off_t,
#         producer_idx_t, producer_off_t,
#         target_t
#     )

# def eval_permuted_other(g):
#     return eval_nll(
#         image_t, permute_rows(other_t, g),
#         studio_idx_t, studio_off_t,
#         producer_idx_t, producer_off_t,
#         target_t
#     )

# def eval_permuted_studios(g):
#     perm = torch.randperm(len(test_dataset), generator=g, device=device).cpu()
#     permuted_batch = [test_dataset[i] for i in perm.tolist()]
#     p_studio_idx, p_studio_off = create_embedding_bag_inputs(
#         [item[2].tolist() for item in permuted_batch]
#     )
#     return eval_nll(
#         image_t, other_t,
#         p_studio_idx.to(device), p_studio_off.to(device),
#         producer_idx_t, producer_off_t,
#         target_t
#     )

# def eval_permuted_producers(g):
#     perm = torch.randperm(len(test_dataset), generator=g, device=device).cpu()
#     permuted_batch = [test_dataset[i] for i in perm.tolist()]
#     p_producer_idx, p_producer_off = create_embedding_bag_inputs(
#         [item[3].tolist() for item in permuted_batch]
#     )
#     return eval_nll(
#         image_t, other_t,
#         studio_idx_t, studio_off_t,
#         p_producer_idx.to(device), p_producer_off.to(device),
#         target_t
#     )

# block_results["image (PCA embedding)"] = permutation_importance_nll(eval_permuted_image, baseline_nll)
# block_results["tabular (other)"] = permutation_importance_nll(eval_permuted_other, baseline_nll)
# block_results["studios"] = permutation_importance_nll(eval_permuted_studios, baseline_nll)
# block_results["producers"] = permutation_importance_nll(eval_permuted_producers, baseline_nll)

# for name, (mean_d, std_d) in sorted(block_results.items(), key=lambda kv: -kv[1][0]):
#     print(f"{name:25s}  ΔNLL = {mean_d:+.4f}  (± {std_d:.4f})")

In [ ]:
# # --- Per-feature importance within the tabular ('other') branch ---
# # X_other_pre columns line up with the columns of other_t/other_test in order.

# other_feature_names = X_other_pre.columns.tolist()
# assert len(other_feature_names) == other_t.shape[1], "Column count mismatch between X_other_pre and other_t"

# feature_results = {}

# for j, feature_name in enumerate(other_feature_names):

#     def eval_permuted_feature(g, col=j):
#         perm = torch.randperm(other_t.size(0), generator=g, device=device)
#         permuted_other = other_t.clone()
#         permuted_other[:, col] = other_t[perm, col]
#         return eval_nll(
#             image_t, permuted_other,
#             studio_idx_t, studio_off_t,
#             producer_idx_t, producer_off_t,
#             target_t
#         )

#     feature_results[feature_name] = permutation_importance_nll(
#         eval_permuted_feature, baseline_nll, n_repeats=10
#     )

# feature_importance_df = pd.DataFrame(
#     [(name, mean_d, std_d) for name, (mean_d, std_d) in feature_results.items()],
#     columns=["feature", "delta_nll", "delta_nll_std"]
# ).sort_values("delta_nll", ascending=False).reset_index(drop=True)

# feature_importance_df.head(20)

In [ ]:
# # --- Visualize ---

# fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# block_names = list(block_results.keys())
# block_means = [block_results[n][0] for n in block_names]
# block_stds = [block_results[n][1] for n in block_names]
# order = np.argsort(block_means)

# axes[0].barh(
#     [block_names[i] for i in order],
#     [block_means[i] for i in order],
#     xerr=[block_stds[i] for i in order]
# )
# axes[0].set_xlabel("ΔNLL when permuted (higher = more important)")
# axes[0].set_title("Block-level importance")
# axes[0].axvline(0, color="black", linewidth=0.8)

# top_n = 15
# top_features = feature_importance_df.head(top_n).iloc[::-1]

# axes[1].barh(
#     top_features["feature"],
#     top_features["delta_nll"],
#     xerr=top_features["delta_nll_std"]
# )
# axes[1].set_xlabel("ΔNLL when permuted (higher = more important)")
# axes[1].set_title(f"Top {top_n} tabular features")
# axes[1].axvline(0, color="black", linewidth=0.8)

# plt.tight_layout()
# plt.show()

### Fusion Network (without images)

In [ ]:
class FusionNetwork2(nn.Module):

    def __init__(self):
        super().__init__()

        # Tabular branch
        self.other_branch = nn.Sequential(
            nn.Linear(82, 48),
            nn.BatchNorm1d(48),
            nn.ReLU(),
            nn.Dropout(0.4)
        )

        self.studio_embedding = nn.EmbeddingBag(
            num_embeddings=n_studios,
            embedding_dim=8,
            mode='mean'
        )

        self.producer_embedding = nn.EmbeddingBag(
            num_embeddings=n_producers,
            embedding_dim=8,
            mode='mean'
        )

        self.fusion = nn.Sequential(
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 2) 
        )

    def forward(self, other, studio_indices, studio_offsets, producer_indices, producer_offsets):
        # Process branches
        other_features = self.other_branch(other)
        studio_features = self.studio_embedding(studio_indices, studio_offsets)
        producer_features = self.producer_embedding(producer_indices, producer_offsets)
        
        combined = torch.cat([other_features, studio_features, producer_features], dim=1)
        out = self.fusion(combined)

        mean = out[:, 0]
        raw_std = out[:, 1]

        std = torch.nn.functional.softplus(raw_std) + 1e-6
        
        return torch.distributions.Normal(mean, std)

In [ ]:
model2 = FusionNetwork2()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model2 = model2.to(device)


class AnimeDataset2(torch.utils.data.Dataset):

    def __init__(
        self,
        other,
        target,
        studio_indices,
        studio_offsets,
        producer_indices,
        producer_offsets
    ):
        self.other = other
        self.target = target

        self.studio_indices = studio_indices
        self.studio_offsets = studio_offsets

        self.producer_indices = producer_indices
        self.producer_offsets = producer_offsets

    def __len__(self):
        return len(self.target)

    def __getitem__(self, idx):

        # Figure out where this anime's studio list starts
        studio_start = self.studio_offsets[idx]

        if idx + 1 < len(self.studio_offsets):
            studio_end = self.studio_offsets[idx + 1]
        else:
            studio_end = len(self.studio_indices)

        studio_indices = self.studio_indices[
            studio_start:studio_end
        ]

        # Same thing for producers
        producer_start = self.producer_offsets[idx]

        if idx + 1 < len(self.producer_offsets):
            producer_end = self.producer_offsets[idx + 1]
        else:
            producer_end = len(self.producer_indices)

        producer_indices = self.producer_indices[
            producer_start:producer_end
        ]

        return (
            self.other[idx],
            studio_indices,
            producer_indices,
            self.target[idx]
        )

def collate_fn2(batch):

    others = torch.stack([item[0] for item in batch])
    targets = torch.stack([item[3] for item in batch])

    studio_indices = []
    studio_offsets = []

    current_offset = 0

    for item in batch:
        indices = item[1]

        studio_offsets.append(current_offset)

        studio_indices.extend(
            indices.tolist()
        )

        current_offset += len(indices)

    studio_indices = torch.tensor(
        studio_indices,
        dtype=torch.long
    )

    studio_offsets = torch.tensor(
        studio_offsets,
        dtype=torch.long
    )

    producer_indices = []
    producer_offsets = []

    current_offset = 0

    for item in batch:
        indices = item[2]

        producer_offsets.append(current_offset)

        producer_indices.extend(
            indices.tolist()
        )

        current_offset += len(indices)

    producer_indices = torch.tensor(
        producer_indices,
        dtype=torch.long
    )

    producer_offsets = torch.tensor(
        producer_offsets,
        dtype=torch.long
    )

    return (
        others,
        studio_indices,
        studio_offsets,
        producer_indices,
        producer_offsets,
        targets
    )

In [ ]:
train_dataset = AnimeDataset2(
    other_train,
    score_train,
    studio_indices_train,
    studio_offsets_train,
    producer_indices_train,
    producer_offsets_train
)

val_dataset = AnimeDataset2(
    other_val,
    score_val,
    studio_indices_val,
    studio_offsets_val,
    producer_indices_val,
    producer_offsets_val
)

test_dataset = AnimeDataset2(
    other_test,
    score_test,
    studio_indices_test,
    studio_offsets_test,
    producer_indices_test,
    producer_offsets_test
)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn2
)


In [ ]:
# full_dataset = ConcatDataset([train_dataset, val_dataset])
# n_samples = len(full_dataset)

# k = 5
# kfold = KFold(n_splits=k, shuffle=True, random_state=42)

# batch_size = train_loader.batch_size
# collate_fn = train_loader.collate_fn
# patience = 15
# n_epochs = 300

# fold_results = []          
# fold_model_states = []    

# for fold, (train_idx, val_idx) in enumerate(kfold.split(np.arange(n_samples))):

#     print(f"\n===== Fold {fold + 1}/{k} =====")

#     model2 = FusionNetwork2().to(device)  
#     optimizer = torch.optim.AdamW(
#         model2.parameters(),
#         lr=2e-4,
#         weight_decay=1e-2
#     )

#     train_subset = Subset(full_dataset, train_idx)
#     val_subset = Subset(full_dataset, val_idx)

#     fold_train_loader = DataLoader(
#         train_subset,
#         batch_size=batch_size,
#         shuffle=True,
#         collate_fn=collate_fn2
#     )
#     fold_val_loader = DataLoader(
#         val_subset,
#         batch_size=batch_size,
#         shuffle=False,
#         collate_fn=collate_fn2
#     )

#     best_val_loss = float("inf")
#     patience_counter = 0
#     best_model_state = None

#     for epoch in range(n_epochs):

#         model2.train()
#         train_loss = 0

#         for (
#             other,
#             studio_indices, studio_offsets,
#             producer_indices, producer_offsets,
#             target
#         ) in fold_train_loader:

#             other = other.to(device)
#             studio_indices = studio_indices.to(device)
#             studio_offsets = studio_offsets.to(device)
#             producer_indices = producer_indices.to(device)
#             producer_offsets = producer_offsets.to(device)
#             target = target.to(device)

#             prediction = model2(
#                 other,
#                 studio_indices, studio_offsets,
#                 producer_indices, producer_offsets
#             )

#             loss = nll_loss(prediction, target)

#             optimizer.zero_grad()
#             loss.backward()
#             optimizer.step()

#             train_loss += loss.item()

#         avg_train_loss = train_loss / len(fold_train_loader)

#         model2.eval()
#         val_loss = 0

#         with torch.no_grad():
#             for (
#                 other,
#                 studio_indices, studio_offsets,
#                 producer_indices, producer_offsets,
#                 target
#             ) in fold_val_loader:

#                 other = other.to(device)
#                 studio_indices = studio_indices.to(device)
#                 studio_offsets = studio_offsets.to(device)
#                 producer_indices = producer_indices.to(device)
#                 producer_offsets = producer_offsets.to(device)
#                 target = target.to(device)

#                 prediction = model2(
#                     other,
#                     studio_indices, studio_offsets,
#                     producer_indices, producer_offsets
#                 )

#                 loss = nll_loss(prediction, target)
#                 val_loss += loss.item()

#         avg_val_loss = val_loss / len(fold_val_loader)

#         if avg_val_loss < best_val_loss:
#             best_val_loss = avg_val_loss
#             patience_counter = 0
#             best_model_state = copy.deepcopy(model2.state_dict())
#         else:
#             patience_counter += 1
#             if patience_counter >= patience:
#                 print(f"Fold {fold + 1} early stopping at epoch {epoch}.")
#                 break

#         if epoch % 5 == 0 or patience_counter == 0:
#             print(
#                 f"  Epoch {epoch}: "
#                 f"Train Loss: {avg_train_loss:.4f} | "
#                 f"Val Loss: {avg_val_loss:.4f}"
#             )

#     print(f"Fold {fold + 1} best val loss: {best_val_loss:.4f}")
#     fold_results.append(best_val_loss)
#     fold_model_states.append(best_model_state)

# fold_results = np.array(fold_results)
# print(f"\n===== CV Results ({k}-fold) =====")
# print(f"Per-fold val loss: {fold_results}")
# print(f"Mean: {fold_results.mean():.4f}  |  Std: {fold_results.std():.4f}")

# best_fold_idx = fold_results.argmin()
# best_model_state = fold_model_states[best_fold_idx]
# model2 = FusionNetwork2().to(device)
# model2.load_state_dict(best_model_state)
# print(f"\nLoaded weights from fold {best_fold_idx + 1} (val loss {fold_results[best_fold_idx]:.4f})")

In [ ]:
# model2.eval()

# total_nll = 0
# total_samples = 0

# all_predictions = []
# all_targets = []

# with torch.no_grad():

#     for (
#         other,
#         studio_indices,
#         studio_offsets,
#         producer_indices,
#         producer_offsets,
#         target
#     ) in test_loader:

#         other = other.to(device)

#         studio_indices = studio_indices.to(device)
#         studio_offsets = studio_offsets.to(device)

#         producer_indices = producer_indices.to(device)
#         producer_offsets = producer_offsets.to(device)

#         target = target.to(device)

#         predictions = model2(
#             other,
#             studio_indices,
#             studio_offsets,
#             producer_indices,
#             producer_offsets
#         )

#         nll = nll_loss(predictions, target)

#         total_nll += nll * 32
#         total_samples += target.size(0)



# final_nll = (
#     total_nll /
#     total_samples
# )


# print(f"Test NLL:  {final_nll:.4f}")

One can see that the images produce a lot of noise. In hindsight, it doesn't really make sense for thumbnails to sustain viewers. It's just clickbait at most and will only give you Week 1 points.  

Now let's do some feature importance.

In [ ]:
def get_full_test_batch2(dataset):
     """Pull the entire test_dataset through the existing collate_fn in one shot."""
     items = [dataset[i] for i in range(len(dataset))]
     return collate_fn2(items)

In [ ]:
# (
#     other_t,
#     studio_idx_t, studio_off_t,
#     producer_idx_t, producer_off_t,
#     target_t
# ) = get_full_test_batch2(test_dataset)

# other_t = other_t.to(device)
# studio_idx_t = studio_idx_t.to(device)
# studio_off_t = studio_off_t.to(device)
# producer_idx_t = producer_idx_t.to(device)
# producer_off_t = producer_off_t.to(device)
# target_t = target_t.to(device)

# model2.eval()

@torch.no_grad()
def eval_nll2(other, studio_idx, studio_off, producer_idx, producer_off, target):
    dist = model2(other, studio_idx, studio_off, producer_idx, producer_off)
    return nll_loss(dist, target).item()

# baseline_nll = eval_nll2(
#     other_t,
#     studio_idx_t, studio_off_t,
#     producer_idx_t, producer_off_t,
#     target_t
# )
# print(f"Baseline test NLL: {baseline_nll:.4f}")

In [ ]:
# --- Block-level importance: image branch, tabular branch, studios, producers ---

block_results = {}

def eval_permuted_other(g):
    return eval_nll2(
        permute_rows(other_t, g),
        studio_idx_t, studio_off_t,
        producer_idx_t, producer_off_t,
        target_t
    )

def eval_permuted_studios(g):
    perm = torch.randperm(len(test_dataset), generator=g, device=device).cpu()
    permuted_batch = [test_dataset[i] for i in perm.tolist()]
    p_studio_idx, p_studio_off = create_embedding_bag_inputs(
        [item[1].tolist() for item in permuted_batch]
    )
    return eval_nll2(
        other_t,
        p_studio_idx.to(device), p_studio_off.to(device),
        producer_idx_t, producer_off_t,
        target_t
    )

def eval_permuted_producers(g):
    perm = torch.randperm(len(test_dataset), generator=g, device=device).cpu()
    permuted_batch = [test_dataset[i] for i in perm.tolist()]
    p_producer_idx, p_producer_off = create_embedding_bag_inputs(
        [item[2].tolist() for item in permuted_batch]
    )
    return eval_nll2(
        other_t,
        studio_idx_t, studio_off_t,
        p_producer_idx.to(device), p_producer_off.to(device),
        target_t
    )

# block_results["tabular (other)"] = permutation_importance_nll(eval_permuted_other, baseline_nll)
# block_results["studios"] = permutation_importance_nll(eval_permuted_studios, baseline_nll)
# block_results["producers"] = permutation_importance_nll(eval_permuted_producers, baseline_nll)

# for name, (mean_d, std_d) in sorted(block_results.items(), key=lambda kv: -kv[1][0]):
#     print(f"{name:25s}  ΔNLL = {mean_d:+.4f}  (± {std_d:.4f})")

In [ ]:
# # --- Per-feature importance within the tabular ('other') branch ---
# # X_other_pre columns line up with the columns of other_t/other_test in order.

# other_feature_names = X_other_pre.columns.tolist()
# assert len(other_feature_names) == other_t.shape[1], "Column count mismatch between X_other_pre and other_t"

# feature_results = {}

# for j, feature_name in enumerate(other_feature_names):

#     def eval_permuted_feature(g, col=j):
#         perm = torch.randperm(other_t.size(0), generator=g, device=device)
#         permuted_other = other_t.clone()
#         permuted_other[:, col] = other_t[perm, col]
#         return eval_nll2(
#             permuted_other,
#             studio_idx_t, studio_off_t,
#             producer_idx_t, producer_off_t,
#             target_t
#         )

#     feature_results[feature_name] = permutation_importance_nll(
#         eval_permuted_feature, baseline_nll, n_repeats=10
#     )

# feature_importance_df = pd.DataFrame(
#     [(name, mean_d, std_d) for name, (mean_d, std_d) in feature_results.items()],
#     columns=["feature", "delta_nll", "delta_nll_std"]
# ).sort_values("delta_nll", ascending=False).reset_index(drop=True)

# feature_importance_df.head(20)

In [ ]:
# # --- Visualize ---

# fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# block_names = list(block_results.keys())
# block_means = [block_results[n][0] for n in block_names]
# block_stds = [block_results[n][1] for n in block_names]
# order = np.argsort(block_means)

# axes[0].barh(
#     [block_names[i] for i in order],
#     [block_means[i] for i in order],
#     xerr=[block_stds[i] for i in order]
# )
# axes[0].set_xlabel("ΔNLL when permuted (higher = more important)")
# axes[0].set_title("Block-level importance")
# axes[0].axvline(0, color="black", linewidth=0.8)

# top_n = 15
# top_features = feature_importance_df.head(top_n).iloc[::-1]

# axes[1].barh(
#     top_features["feature"],
#     top_features["delta_nll"],
#     xerr=top_features["delta_nll_std"]
# )
# axes[1].set_xlabel("ΔNLL when permuted (higher = more important)")
# axes[1].set_title(f"Top {top_n} tabular features")
# axes[1].axvline(0, color="black", linewidth=0.8)

# plt.tight_layout()
# plt.savefig("../readme/score_importance.png")
# plt.show()

### Fusion Network (only tabular data)

In [ ]:
class FusionNetwork3(nn.Module):
    def __init__(self, init_target_std=1.0):
        super().__init__()

        self.input_norm = nn.LayerNorm(70)
        
        self.fc1 = nn.Linear(70, 64)
        self.act1 = nn.GELU() 
        self.drop1 = nn.Dropout(0.15)
        
        self.fc2 = nn.Linear(64, 32)
        self.act2 = nn.GELU()
        self.drop2 = nn.Dropout(0.10)
        
        self.mean_head = nn.Linear(32, 1)
        self.std_head = nn.Linear(32, 1)
        
        initial_bias = torch.log(torch.exp(torch.tensor(init_target_std)) - 1.0)
        nn.init.constant_(self.std_head.bias, initial_bias)

    def forward(self, other):
        if other.dim() == 1:
            other = other.unsqueeze(0)
            
        x = self.input_norm(other)
        x = self.drop1(self.act1(self.fc1(x)))
        x = self.drop2(self.act2(self.fc2(x)))
        
        mean = self.mean_head(x).squeeze(-1)
        raw_std = self.std_head(x).squeeze(-1)
        
        std = F.softplus(raw_std) + 1e-4
        
        return torch.distributions.Normal(mean, std)

In [ ]:
model3 = FusionNetwork3()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model3 = model3.to(device)

In [ ]:
train_dataset3 = TensorDataset(other_train, score_train)
val_dataset3 = TensorDataset(other_val, score_val)

full_dataset = ConcatDataset([train_dataset3, val_dataset3])
n_samples = len(full_dataset)

k = 5
kfold = KFold(n_splits=k, shuffle=True, random_state=42)

patience = 15
n_epochs = 300
batch_size = 64

fold_results = []        
fold_model_states = []   
fold_best_epochs = []

optimizer = torch.optim.AdamW(
    model3.parameters(),
    lr=5e-4,
    weight_decay=1e-4
)

scheduler = CosineAnnealingWarmRestarts(
    optimizer,
    T_0=30,     
    T_mult=1,   
    eta_min=1e-6
)

for fold, (train_idx, val_idx) in enumerate(kfold.split(np.arange(n_samples))):

    print(f"\n===== Fold {fold + 1}/{k} =====")

    model3 = FusionNetwork3().to(device)  
    
    optimizer = torch.optim.AdamW(
        model3.parameters(),
        lr=5e-4,
        weight_decay=1e-4
    )

    train_subset = Subset(full_dataset, train_idx)
    val_subset = Subset(full_dataset, val_idx)

    fold_train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, drop_last=True)
    fold_val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)

    best_val_loss = float("inf")
    patience_counter = 0
    best_model_state = None
    best_epoch = 0 

    for epoch in range(n_epochs):

        model3.train()
        train_loss = 0

        for other, *embeddings, target in fold_train_loader:

            other = other.to(device)
            target = target.to(device)

            prediction = model3(other)
            loss = nll_loss(prediction, target)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model3.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss += loss.item()

        avg_train_loss = train_loss / len(fold_train_loader)

        model3.eval()
        val_loss = 0

        with torch.no_grad():
            for other, *embeddings, target in fold_val_loader:

                other = other.to(device)
                target = target.to(device)

                prediction = model3(other)
                loss = nll_loss(prediction, target)
                val_loss += loss.item()

        avg_val_loss = val_loss / len(fold_val_loader)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_epoch = epoch
            patience_counter = 0
            best_model_state = copy.deepcopy(model3.state_dict())
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Fold {fold + 1} early stopping at epoch {epoch}. Best epoch was {best_epoch}.")
                break

        if epoch % 5 == 0 or patience_counter == 0:
            print(
                f"  Epoch {epoch}: "
                f"Train Loss: {avg_train_loss:.4f} | "
                f"Val Loss: {avg_val_loss:.4f}"
            )

        scheduler.step()

    print(f"Fold {fold + 1} best val loss: {best_val_loss:.4f} at epoch {best_epoch}")
    fold_results.append(best_val_loss)
    fold_model_states.append(best_model_state)
    fold_best_epochs.append(best_epoch)

fold_results = np.array(fold_results)
fold_best_epochs = np.array(fold_best_epochs)

print(f"\n===== CV Results ({k}-fold) =====")
print(f"Per-fold val loss: {fold_results}")
print(f"Mean loss: {fold_results.mean():.4f}  |  Std loss: {fold_results.std():.4f}")
print(f"Per-fold best epochs: {fold_best_epochs}")

avg_best_epoch = int(np.round(fold_best_epochs.mean())) + 1
print(f"Target epochs for full retraining: {avg_best_epoch}")

In [ ]:
full_train_loader = DataLoader(
    full_dataset, 
    batch_size=batch_size, 
    shuffle=True, 
    drop_last=True
)

final_model = FusionNetwork3().to(device)

final_optimizer = torch.optim.AdamW(
    final_model.parameters(),
    lr=5e-4,
    weight_decay=1e-4
)

final_scheduler = CosineAnnealingLR(
    final_optimizer, 
    T_max=avg_best_epoch, 
    eta_min=1e-6
)

print(f"\n===== Retraining Final Model on 100% Data ({avg_best_epoch} Epochs) =====")

for epoch in range(avg_best_epoch):
    final_model.train()
    train_loss = 0.0

    for batch in full_train_loader:
        other = batch[0].to(device)
        target = batch[-1].to(device)

        prediction = final_model(other)
        loss = nll_loss(prediction, target)

        final_optimizer.zero_grad()
        loss.backward()
        final_optimizer.step()

        train_loss += loss.item()

    final_scheduler.step()
    avg_train_loss = train_loss / len(full_train_loader)

    if (epoch + 1) % 5 == 0 or (epoch + 1) == avg_best_epoch:
        print(f"Epoch [{epoch + 1}/{avg_best_epoch}] | Full Train Loss: {avg_train_loss:.4f}")

torch.save(final_model.state_dict(), "fusion_network_final.pt")
print("\nFinal model training complete. Checkpoint saved to 'fusion_network_final.pt'.")

In [ ]:
test_dataset3 = TensorDataset(other_test, score_test)

test_loader = DataLoader(
    test_dataset3,
    batch_size=batch_size,
    shuffle=False
)


In [ ]:
final_model.eval()

total_nll = 0
total_samples = 0

all_predictions = []
all_targets = []

with torch.no_grad():

    for (
        other,
        target
    ) in test_loader:

        other = other.to(device)

        target = target.to(device)

        predictions = final_model(other)
        
        nll = nll_loss(predictions, target)

        total_nll += nll * 64
        total_samples += target.size(0)



final_nll = (
    total_nll /
    total_samples
)


print(f"Test NLL:  {final_nll:.4f}")

### Feature Importance (FusionNetwork3)

In [ ]:
final_model.eval()

@torch.no_grad()
def eval_nll3(other, target):
    dist = final_model(other)
    return nll_loss(dist, target).item()

other_t3 = other_test.to(device)
target_t3 = score_test.to(device)

baseline_nll3 = eval_nll3(other_t3, target_t3)
print(f"Baseline test NLL (FusionNetwork3): {baseline_nll3:.4f}")


In [ ]:
# --- Per-feature importance within the tabular ('other') branch (FusionNetwork3) ---
# There's no image/studio/producer block to permute here, since FusionNetwork3
# only consumes the tabular 'other' features, so we go straight to per-feature.

other_feature_names3 = X_other_pre.columns.tolist()
assert len(other_feature_names3) == other_t3.shape[1], "Column count mismatch between X_other_pre and other_t3"

feature_results3 = {}

for j, feature_name in enumerate(other_feature_names3):

    def eval_permuted_feature3(g, col=j):
        perm = torch.randperm(other_t3.size(0), generator=g, device=device)
        permuted_other = other_t3.clone()
        permuted_other[:, col] = other_t3[perm, col]
        return eval_nll3(permuted_other, target_t3)

    feature_results3[feature_name] = permutation_importance_nll(
        eval_permuted_feature3, baseline_nll3, n_repeats=10
    )

feature_importance_df3 = pd.DataFrame(
    [(name, mean_d, std_d) for name, (mean_d, std_d) in feature_results3.items()],
    columns=["feature", "delta_nll", "delta_nll_std"]
).sort_values("delta_nll", ascending=False).reset_index(drop=True)

feature_importance_df3.head(20)


In [ ]:
top_n = 15
top_features3 = feature_importance_df3.head(top_n).iloc[::-1]

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(
    top_features3["feature"],
    top_features3["delta_nll"],
    xerr=top_features3["delta_nll_std"]
)
ax.set_xlabel("ΔNLL when permuted (higher = more important)")
ax.set_title(f"Top {top_n} tabular features (FusionNetwork3)")
ax.axvline(0, color="black", linewidth=0.8)

plt.tight_layout()
plt.savefig("../readme/forum_importance_fusion3.png")
plt.show()


### Current Anime Prediction

In [ ]:
final_model.eval()

real_other_t = real_other.to(device)

with torch.no_grad():
    output = final_model(real_other_t)
    means = output.mean.cpu().numpy()
    stds = output.stddev.cpu().numpy()

plt.hist(stds)
plt.title("Uncertainties")
plt.show()


In [ ]:
predictions = {}
for index, title in enumerate(real_df['title']):
    predictions[title] = [float(means[index]), float(stds[index])]

print(predictions)

In [ ]:
sorted_predictions = dict(sorted(predictions.items(), key=lambda item: item[1][0], reverse=True))
pprint(sorted_predictions, indent=4, sort_dicts=False)

In [ ]:
target_dir = "..\data\processed"
file_name = "forum_predictions.json" # change depending on what your model is training on
file_path = os.path.join(target_dir, file_name)

with open(file_path, "w", encoding="utf-8") as file:
    json.dump(sorted_predictions, file, indent=4)